<a href="https://github.com/gutris1/segsmaker-fast">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

*   get your civitai api key from [here](https://civitai.com/user/account)
*   get your hugging face token from [here](https://huggingface.co/settings/tokens)

In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}
#@markdown Initial setup, API keys, and Google Drive mounting.

Webui = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai__Key = '' # @param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = '' # @param { type: "string", placeholder: "Your Huggingface READ Token (optional)" }
Mount_GDrive = 'No' # @param ["Yes", "No"]

mount = Mount_GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/gutris1/segsmaker-fast/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai__Key" --hf_read_token="$HF_Read_Token"

if mount == 'Yes':
    from pathlib import Path

    d = Path('/content/drive/MyDrive/Segsmaker')

    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        if p is not None:
            f = d / n
            f.mkdir(parents=True, exist_ok=True)
            s = p / f'drive-{n}'
            if not s.exists():
                s.symlink_to(f, target_is_directory=True)

    if 'WebUI_Output' in globals() and WebUI_Output is not None:
        try:
            !rm -rf $WebUI_Output
            o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
            o.mkdir(parents=True, exist_ok=True)
            WebUI_Output.symlink_to(o, target_is_directory=True)
        except Exception as e:
            print(f"Error mapping output: {e}")

    if Webui not in {'ComfyUI', 'SwarmUI'} and 'WebUI' in globals() and WebUI is not None:
        try:
            wc = WebUI / 'cache'
            !rm -rf $wc
            c = d / 'cache'
            c.mkdir(parents=True, exist_ok=True)
            wc.symlink_to(c, target_is_directory=True)
        except Exception as e:
            print(f"Error mapping cache: {e}")

In [ ]:
# @title <b><font color='orange'>Model Downloader - 5 Checkpoint + 5 LoRA + VAE</font></b> {"display-mode":"form"}
#@markdown Download checkpoints, LoRAs, and VAEs to Colab or persistent Google Drive.

#@markdown <details><summary><b>Checkpoints</b></summary>
Checkpoint_1 = 'https://huggingface.co/pantat88/back_up/resolve/main/bigblu25dmix25DStyle_v10.safetensors' # @param { type: "string", placeholder: "Checkpoint URL 1" }
Checkpoint_2 = '' # @param { type: "string", placeholder: "Checkpoint URL 2" }
Checkpoint_3 = '' # @param { type: "string", placeholder: "Checkpoint URL 3" }
Checkpoint_4 = '' # @param { type: "string", placeholder: "Checkpoint URL 4" }
Checkpoint_5 = '' # @param { type: "string", placeholder: "Checkpoint URL 5" }
#@markdown </details>

#@markdown <details><summary><b>LoRA</b></summary>
Lora_1 = 'https://civitai.com/models/122359/detail-tweaker-xl' # @param { type: "string", placeholder: "Lora URL 1" }
Lora_2 = 'https://civitai.com/models/669571/pony-add-more-details details-add-more-pony.safetensors' # @param { type: "string", placeholder: "Lora URL 2" }
Lora_3 = '' # @param { type: "string", placeholder: "Lora URL 3" }
Lora_4 = '' # @param { type: "string", placeholder: "Lora URL 4" }
Lora_5 = '' # @param { type: "string", placeholder: "Lora URL 5" }
#@markdown </details>

#@markdown <details><summary><b>VAE</b></summary>
VAE_URL = '' # @param { type: "string", placeholder: "VAE URL or leave empty" }
#@markdown </details>

#@markdown <details><summary><b>Speed & Storage Options</b></summary>
Parallel_Download = True # @param { type: "boolean" }
Max_Workers = 3 # @param { type: "slider", min: 1, max: 5, step: 1 }
Load_From_Drive = False # @param { type: "boolean" }
#@markdown </details>

import os
from pathlib import Path

ckpt_dest = CKPT
lora_dest = LORA
vae_dest = VAE

if Load_From_Drive:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        try:
            drive.mount('/content/drive')
        except Exception as e:
            print(f"Error mounting Google Drive: {e}")

    if os.path.exists('/content/drive/MyDrive'):
        d = Path('/content/drive/MyDrive/Segsmaker')
        d.mkdir(parents=True, exist_ok=True)

        ckpt_dest = d / 'checkpoint'
        lora_dest = d / 'lora'
        vae_dest = d / 'vae'

        ckpt_dest.mkdir(exist_ok=True)
        lora_dest.mkdir(exist_ok=True)
        vae_dest.mkdir(exist_ok=True)

        for name, p, f in [('checkpoint', CKPT, ckpt_dest), ('lora', LORA, lora_dest), ('vae', VAE, vae_dest)]:
            if p is not None:
                s = p / f'drive-{name}'
                if not s.exists():
                    s.symlink_to(f, target_is_directory=True)
        print("Redirecting downloads to persistent Google Drive folders.")

downloads = []
for ckpt in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    if ckpt.strip():
        downloads.append(f"{ckpt.strip()} {ckpt_dest}")

for lora in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    if lora.strip():
        downloads.append(f"{lora.strip()} {lora_dest}")

if VAE_URL.strip():
    downloads.append(f"{VAE_URL.strip()} {vae_dest}")

if downloads:
    if Parallel_Download:
        parallel_download_files(downloads, max_workers=Max_Workers)
    else:
        for item in downloads:
            %download $item
else:
    print("No models or loras selected for download.")

In [ ]:
# @title <b><font color='orange'>Extra Assets - Extensions, Embeddings, Upscalers</font></b> {"display-mode":"form"}
#@markdown Download extra packages, texture embeddings, or high-res upscalers.

#@markdown <details><summary><b>Extensions / ComfyUI Custom Nodes</b></summary>
Extension_1 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
Extension_2 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
Extension_3 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
Extension_4 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
Extension_5 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
#@markdown </details>

#@markdown <details><summary><b>Embeddings</b></summary>
Embedding_1 = '' # @param { type: "string", placeholder: "Embedding URL or leave empty" }
Embedding_2 = '' # @param { type: "string", placeholder: "Embedding URL or leave empty" }
Embedding_3 = '' # @param { type: "string", placeholder: "Embedding URL or leave empty" }
#@markdown </details>

#@markdown <details><summary><b>Upscalers</b></summary>
Upscaler_1 = '' # @param { type: "string", placeholder: "Upscaler URL or leave empty" }
Upscaler_2 = '' # @param { type: "string", placeholder: "Upscaler URL or leave empty" }
Upscaler_3 = '' # @param { type: "string", placeholder: "Upscaler URL or leave empty" }
#@markdown </details>

#@markdown <details><summary><b>Speed Options</b></summary>
Assets_Parallel_Download = True # @param { type: "boolean" }
Assets_Max_Workers = 3 # @param { type: "slider", min: 1, max: 5, step: 1 }
#@markdown </details>

# Clone extensions
extensions = [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5]
for ext in extensions:
    if ext.strip():
        %cd -q $Extensions
        %clone $ext

# Download Embeddings & Upscalers
downloads = []
for emb in [Embedding_1, Embedding_2, Embedding_3]:
    if emb.strip():
        downloads.append(f"{emb.strip()} {Embeddings}")

for ups in [Upscaler_1, Upscaler_2, Upscaler_3]:
    if ups.strip():
        downloads.append(f"{ups.strip()} {Upscalers}")

if downloads:
    if Assets_Parallel_Download:
        parallel_download_files(downloads, max_workers=Assets_Max_Workers)
    else:
        for item in downloads:
            %download $item
else:
    print("No extra embeddings or upscalers selected.")

In [ ]:
# @title <b><font color='orange'>FLUX Model Downloader</font></b> {"display-mode":"form"}
#@markdown Download component weights for the FLUX model family.

#@markdown <details><summary><b>FLUX Variant</b></summary>
FLUX_Variant = 'FLUX.1-schnell' # @param ["FLUX.1-schnell", "FLUX.1-dev"]
#@markdown </details>

#@markdown <details><summary><b>Component URLs</b></summary>
FLUX_Unet = '' # @param { type: "string", placeholder: "FLUX Unet URL or leave empty" }
FLUX_Clip_L = '' # @param { type: "string", placeholder: "FLUX Clip L URL or leave empty" }
FLUX_T5XXL = '' # @param { type: "string", placeholder: "FLUX T5XXL URL or leave empty" }
FLUX_VAE = '' # @param { type: "string", placeholder: "FLUX VAE URL or leave empty" }
#@markdown </details>

#@markdown <details><summary><b>Speed Options</b></summary>
Parallel_FLUX_Download = True # @param { type: "boolean" }
FLUX_Max_Workers = 2 # @param { type: "slider", min: 1, max: 3, step: 1 }
#@markdown </details>

default_urls = {
    'FLUX.1-schnell': {
        'unet': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/flux1-schnell.safetensors',
        'clip': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        't5': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
        'vae': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors'
    },
    'FLUX.1-dev': {
        'unet': 'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/flux1-dev.safetensors',
        'clip': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        't5': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
        'vae': 'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors'
    }
}

u_url = FLUX_Unet.strip() or default_urls[FLUX_Variant]['unet']
c_url = FLUX_Clip_L.strip() or default_urls[FLUX_Variant]['clip']
t5_url = FLUX_T5XXL.strip() or default_urls[FLUX_Variant]['t5']
vae_url = FLUX_VAE.strip() or default_urls[FLUX_Variant]['vae']

downloads = []
if u_url: downloads.append(f"{u_url} {UNET}")
if c_url: downloads.append(f"{c_url} {CLIP}")
if t5_url: downloads.append(f"{t5_url} {CLIP}")
if vae_url: downloads.append(f"{vae_url} {VAE}")

if downloads:
    if Parallel_FLUX_Download:
        parallel_download_files(downloads, max_workers=FLUX_Max_Workers)
    else:
        for item in downloads:
            %download $item

In [ ]:
''' Controlnet '''
%run $Controlnet_Widget

In [ ]:
# @title <b><font color='orange'>Launcher WebUI</font></b> {"display-mode":"form"}
#@markdown Launch the Stable Diffusion or ComfyUI interface.
#@markdown *Select the same WebUI that you installed in the first cell.*

Software = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]

#@markdown <details><summary><b>Tunnel Tokens</b></summary>
Ngrok_Token = '' # @param { type: "string", placeholder: "Your Ngrok Token (optional)" }
Zrok_Token = '' # @param { type: "string", placeholder: "Your Zrok Token (optional)" }
#@markdown </details>

#@markdown <details><summary><b>Pengaturan Tambahan</b></summary>
Extra_Args = '--xformers --cuda-malloc --cuda-stream' # @param { type: "string", placeholder: "Additional parameters (optional)" }
Skip_ComfyUI_Check = False # @param { type: "boolean" }
Skip_Widget = False # @param { type: "boolean" }
#@markdown </details>

args = Extra_Args.strip()
if Zrok_Token.strip():
    args += f" --Z={Zrok_Token.strip()}"
if Ngrok_Token.strip():
    args += f" --N={Ngrok_Token.strip()}"
if Skip_ComfyUI_Check:
    args += " --skip-comfyui-check"
if Skip_Widget:
    args += " --skip-widget"

%cd -q $WebUI
%run segsmaker.py $args

In [ ]:
# @title <b><font color='orange'>Extras (Utilities & Maintenance)</font></b> {"display-mode":"form"}
#@markdown Maintenance and diagnostic tools.

Utility_Command = 'Check Storage' # @param ["Check Storage", "Zip Output Images", "Register Zrok Account", "Change API Keys", "Uninstall WebUI", "Delete Everything"]

if Utility_Command == 'Check Storage':
    %storage
elif Utility_Command == 'Zip Output Images':
    %cd -q $HOME
    print("Zipping output images...")
    get_ipython().run_cell_magic('zipping', '', 'name=drive_outputs\ninputs=$WebUI_Output\noutputs=$HOME')
elif Utility_Command == 'Register Zrok Account':
    %zrok_register
elif Utility_Command == 'Change API Keys':
    %change_key
elif Utility_Command == 'Uninstall WebUI':
    %uninstall_webui
elif Utility_Command == 'Delete Everything':
    %delete_everything